# INbreast External Validation: Evaluating All Stage 1-4 Models

## Purpose

External validation of all trained models on the independent INbreast dataset to assess generalization.

### Critical: Channel Configuration

| Stage | Input Channels | Normalization | Notes |
|-------|---------------|---------------|-------|
| Stage 1 | 3 (RGB) | ImageNet | Pretrained weights |
| Stage 2 | 1 (Gray) | [0.5] | Modified conv1 |
| Stage 3 | 1 (Gray) | [0.5] | Modified conv1 |
| Stage 4 | 3 (RGB) | ImageNet | Multi-view, uses GrayscaleToRGB |

### Literature References

1. **Moreira et al. (2012)** - "INbreast: Toward a Full-field Digital Mammographic Database"
2. **Kelly et al. (2019)** - "Key challenges for delivering clinical impact with artificial intelligence"
3. **Youden (1950)** - "Index for rating diagnostic tests" - Optimal threshold selection

## 1. Setup and Imports

In [1]:
import os
import sys
import json
import warnings
from dataclasses import dataclass
from datetime import datetime
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image

from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, confusion_matrix,
    precision_score, recall_score, f1_score
)

from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Add project to path
sys.path.insert(0, r'D:\Project')
from baseline_models.models import get_model, CBAM

warnings.filterwarnings('ignore')

print("=" * 70)
print("INBREAST EXTERNAL VALIDATION")
print("=" * 70)
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("=" * 70)

INBREAST EXTERNAL VALIDATION
Python version: 3.13.5
PyTorch version: 2.9.1+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU Memory: 34.2 GB


## 2. Configuration

In [2]:
@dataclass
class ValidationConfig:
    """Configuration for INbreast external validation."""
    inbreast_dir: str = r'D:\Project\data\inbreast_preprocessed'
    view_pairs_csv: str = r'D:\Project\data\inbreast_preprocessed\view_pairs.csv'
    output_dir: str = r'D:\Project\results\inbreast_validation'
    
    batch_size: int = 16
    num_workers: int = 0  # Windows compatibility
    
    img_size_stage123: int = 224
    img_size_stage4: int = 768
    
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    def __post_init__(self):
        os.makedirs(self.output_dir, exist_ok=True)

config = ValidationConfig()

print("\nVALIDATION CONFIGURATION")
print("=" * 70)
print(f"INbreast Directory:   {config.inbreast_dir}")
print(f"View Pairs CSV:       {config.view_pairs_csv}")
print(f"Output Directory:     {config.output_dir}")
print(f"Device:               {config.device}")
print("=" * 70)


VALIDATION CONFIGURATION
INbreast Directory:   D:\Project\data\inbreast_preprocessed
View Pairs CSV:       D:\Project\data\inbreast_preprocessed\view_pairs.csv
Output Directory:     D:\Project\results\inbreast_validation
Device:               cuda


## 3. Model Registry

In [3]:
MODEL_REGISTRY = {
    'Stage1': {
        # Stage 1: Pretrained ImageNet -> 3-channel RGB
        'ResNet34': {'path': r'D:\Project\models\ResNet34_best.pth', 'img_size': 224, 'multiview': False, 'channels': 3},
        'ResNet50': {'path': r'D:\Project\models\ResNet50_best.pth', 'img_size': 224, 'multiview': False, 'channels': 3},
        'VGG16': {'path': r'D:\Project\models\VGG16_best.pth', 'img_size': 224, 'multiview': False, 'channels': 3},
        'DenseNet121': {'path': r'D:\Project\models\DenseNet121_best.pth', 'img_size': 224, 'multiview': False, 'channels': 3},
        'EfficientNet-B0': {'path': r'D:\Project\models\EfficientNet-B0_best.pth', 'img_size': 224, 'multiview': False, 'channels': 3},
        'MobileNetV2': {'path': r'D:\Project\models\MobileNetV2_best.pth', 'img_size': 224, 'multiview': False, 'channels': 3},
    },
    'Stage2': {
        # Stage 2: Modified conv1 -> 1-channel grayscale
        'ResNet50Stage2': {'path': r'D:\Project\models\stage2\ResNet50Stage2_best.pth', 'img_size': 224, 'multiview': False, 'channels': 1},
        'CBAMResNet50': {'path': r'D:\Project\models\stage2\CBAMResNet50_best.pth', 'img_size': 224, 'multiview': False, 'channels': 1},
        'HybridViT': {'path': r'D:\Project\models\stage2\HybridViT_best.pth', 'img_size': 224, 'multiview': False, 'channels': 1},
    },
    'Stage3': {
        # Stage 3: Same as Stage 2 with Optuna-tuned hyperparameters
        'ResNet50Stage2': {'path': r'D:\Project\models\stage3\ResNet50Stage2_t009_best.pth', 'img_size': 224, 'multiview': False, 'channels': 1},
        'CBAMResNet50': {'path': r'D:\Project\models\stage3\CBAMResNet50_t019_best.pth', 'img_size': 224, 'multiview': False, 'channels': 1},
        'HybridViT': {'path': r'D:\Project\models\stage3\HybridViT_t038_best.pth', 'img_size': 224, 'multiview': False, 'channels': 1},
    },
    'Stage4': {
        # Stage 4: Multi-view with 3-channel RGB (GrayscaleToRGB in training)
        'ResNet50Stage2': {'path': r'D:\Project\stage4_results\ResNet50Stage2_best.pt', 'img_size': 768, 'multiview': True, 'channels': 3},
        'CBAMResNet50': {'path': r'D:\Project\stage4_results\CBAMResNet50_best.pt', 'img_size': 768, 'multiview': True, 'channels': 3},
        'HybridViT': {'path': r'D:\Project\stage4_results\HybridViT_best.pt', 'img_size': 768, 'multiview': True, 'channels': 3},
    }
}

print("\nMODEL CHECKPOINT VERIFICATION")
print("=" * 70)
total_models = 0
found_models = 0

for stage, models_dict in MODEL_REGISTRY.items():
    stage_found = sum(1 for m in models_dict.values() if os.path.exists(m['path']))
    stage_total = len(models_dict)
    total_models += stage_total
    found_models += stage_found
    print(f"\n{stage}: {stage_found}/{stage_total} models found")
    for name, info in models_dict.items():
        exists = os.path.exists(info['path'])
        status = "[FOUND]  " if exists else "[MISSING]"
        mv_str = " (Multi-View)" if info['multiview'] else ""
        print(f"  {status} {name} @ {info['img_size']}x{info['img_size']} {info['channels']}ch{mv_str}")

print(f"\n{'='*70}")
print(f"TOTAL: {found_models}/{total_models} models available")
print("=" * 70)


MODEL CHECKPOINT VERIFICATION

Stage1: 6/6 models found
  [FOUND]   ResNet34 @ 224x224 3ch
  [FOUND]   ResNet50 @ 224x224 3ch
  [FOUND]   VGG16 @ 224x224 3ch
  [FOUND]   DenseNet121 @ 224x224 3ch
  [FOUND]   EfficientNet-B0 @ 224x224 3ch
  [FOUND]   MobileNetV2 @ 224x224 3ch

Stage2: 3/3 models found
  [FOUND]   ResNet50Stage2 @ 224x224 1ch
  [FOUND]   CBAMResNet50 @ 224x224 1ch
  [FOUND]   HybridViT @ 224x224 1ch

Stage3: 3/3 models found
  [FOUND]   ResNet50Stage2 @ 224x224 1ch
  [FOUND]   CBAMResNet50 @ 224x224 1ch
  [FOUND]   HybridViT @ 224x224 1ch

Stage4: 3/3 models found
  [FOUND]   ResNet50Stage2 @ 768x768 3ch (Multi-View)
  [FOUND]   CBAMResNet50 @ 768x768 3ch (Multi-View)
  [FOUND]   HybridViT @ 768x768 3ch (Multi-View)

TOTAL: 15/15 models available


## 4. Model Architectures (Matching Stage 4 Training)

In [4]:
# CBAM modules (same as training)
class ChannelAttention(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))

class CBAM_Module(nn.Module):
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction)
        self.spatial_attention = SpatialAttention()
    
    def forward(self, x):
        x = x * self.channel_attention(x)
        x = x * self.spatial_attention(x)
        return x

print("CBAM modules defined!")

CBAM modules defined!


In [5]:
# Stage 4 Model Architectures (EXACTLY matching training - 3-channel RGB)

class ResNet50Stage2_3ch(nn.Module):
    """ResNet50 for Stage 4 (3-channel RGB input)."""
    def __init__(self, num_classes: int = 2, pretrained: bool = True, dropout: float = 0.0):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        backbone = models.resnet50(weights=weights)
        
        # Keep original 3-channel conv1
        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = backbone.avgpool
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


class CBAMResNet50_3ch(nn.Module):
    """CBAM-enhanced ResNet50 for Stage 4 (3-channel RGB input)."""
    def __init__(self, num_classes: int = 2, pretrained: bool = True, dropout: float = 0.0):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        resnet = models.resnet50(weights=weights)
        
        self.conv1 = resnet.conv1
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        
        self.layer1 = resnet.layer1
        self.cbam1 = CBAM_Module(256)
        self.layer2 = resnet.layer2
        self.cbam2 = CBAM_Module(512)
        self.layer3 = resnet.layer3
        self.cbam3 = CBAM_Module(1024)
        self.layer4 = resnet.layer4
        self.cbam4 = CBAM_Module(2048)
        
        self.avgpool = resnet.avgpool
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(2048, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.cbam1(self.layer1(x))
        x = self.cbam2(self.layer2(x))
        x = self.cbam3(self.layer3(x))
        x = self.cbam4(self.layer4(x))
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


class HybridViT_3ch(nn.Module):
    """Hybrid ViT for Stage 4 (3-channel RGB input, 768x768)."""
    def __init__(self, num_classes: int = 2, img_size: int = 768, embed_dim: int = 384,
                 num_heads: int = 6, num_layers: int = 4, mlp_ratio: int = 4, dropout: float = 0.0):
        super().__init__()
        self.img_size = img_size
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        self.cnn_backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3
        )
        
        cnn_out_channels = 1024
        feature_map_size = img_size // 16
        
        self.patch_embed = nn.Conv2d(cnn_out_channels, embed_dim, kernel_size=1, stride=1)
        num_patches = feature_map_size ** 2
        
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * mlp_ratio,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.norm = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(embed_dim, num_classes)
        
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.cnn_backbone(x)
        x = self.patch_embed(x)
        x = x.flatten(2).transpose(1, 2)
        
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        x = x + self.pos_embed
        
        x = self.transformer(x)
        x = self.norm(x[:, 0])
        x = self.dropout(x)
        x = self.head(x)
        return x

print("Stage 4 model architectures defined (3-channel RGB)!")

Stage 4 model architectures defined (3-channel RGB)!


In [6]:
# Multi-View Wrapper (EXACTLY matching Stage 4 training - 3-channel RGB)

class MultiViewWrapper_3ch(nn.Module):
    """
    Multi-view wrapper for Stage 4 models (3-channel RGB input).
    Matches the training architecture exactly.
    """
    def __init__(self, base_model_name: str, num_classes: int = 2, 
                 dropout: float = 0.0, img_size: int = 768, fusion: str = 'concat'):
        super().__init__()
        self.fusion = fusion
        self.base_model_name = base_model_name
        
        if base_model_name == 'ResNet50Stage2':
            self.backbone = self._create_resnet_backbone(dropout)
            self.feature_dim = 2048
        elif base_model_name == 'CBAMResNet50':
            self.backbone = self._create_cbam_backbone(dropout)
            self.feature_dim = 2048
        elif base_model_name == 'HybridViT':
            self.backbone = self._create_vit_backbone(dropout, img_size)
            self.feature_dim = 384
        else:
            raise ValueError(f"Unknown model: {base_model_name}")
        
        if fusion == 'concat':
            self.classifier = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(self.feature_dim * 2, num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(self.feature_dim, num_classes)
            )
    
    def _create_resnet_backbone(self, dropout):
        """Create ResNet50 backbone (3-channel, matching training)."""
        weights = models.ResNet50_Weights.IMAGENET1K_V1
        backbone = models.resnet50(weights=weights)
        # Keep original 3-channel conv1 - DO NOT modify!
        backbone.fc = nn.Identity()
        return backbone
    
    def _create_cbam_backbone(self, dropout):
        """Create CBAM ResNet50 backbone (3-channel, matching training)."""
        model = CBAMResNet50_3ch(num_classes=2, pretrained=True, dropout=dropout)
        model.fc = nn.Identity()
        return model
    
    def _create_vit_backbone(self, dropout, img_size):
        """Create HybridViT backbone (3-channel, matching training)."""
        model = HybridViT_3ch(num_classes=2, img_size=img_size, dropout=dropout)
        return model
    
    def extract_features(self, x):
        """Extract features from backbone."""
        if self.base_model_name == 'HybridViT':
            # For HybridViT, run through CNN backbone and get features before head
            B = x.shape[0]
            x = self.backbone.cnn_backbone(x)
            x = self.backbone.patch_embed(x)
            x = x.flatten(2).transpose(1, 2)
            cls_tokens = self.backbone.cls_token.expand(B, -1, -1)
            x = torch.cat([cls_tokens, x], dim=1)
            x = x + self.backbone.pos_embed
            x = self.backbone.transformer(x)
            x = self.backbone.norm(x[:, 0])
            return x
        elif self.base_model_name == 'CBAMResNet50':
            x = self.backbone.conv1(x)
            x = self.backbone.bn1(x)
            x = self.backbone.relu(x)
            x = self.backbone.maxpool(x)
            x = self.backbone.cbam1(self.backbone.layer1(x))
            x = self.backbone.cbam2(self.backbone.layer2(x))
            x = self.backbone.cbam3(self.backbone.layer3(x))
            x = self.backbone.cbam4(self.backbone.layer4(x))
            x = self.backbone.avgpool(x)
            x = torch.flatten(x, 1)
            return x
        else:
            # Standard ResNet
            x = self.backbone(x)
            if x.dim() > 2:
                x = F.adaptive_avg_pool2d(x, 1).flatten(1)
            return x
    
    def forward(self, cc_img, mlo_img):
        cc_feat = self.extract_features(cc_img)
        mlo_feat = self.extract_features(mlo_img)
        
        if self.fusion == 'concat':
            fused = torch.cat([cc_feat, mlo_feat], dim=1)
        else:
            fused = (cc_feat + mlo_feat) / 2
        
        return self.classifier(fused)

print("MultiViewWrapper_3ch defined (matching Stage 4 training)!")

MultiViewWrapper_3ch defined (matching Stage 4 training)!


## 5. Dataset Classes

In [7]:
class INbreastDataset(Dataset):
    """Single-view dataset with channel support."""
    def __init__(self, data_dir: str, transform=None, channels: int = 1):
        self.data_dir = data_dir
        self.transform = transform
        self.channels = channels
        self.samples = []
        
        for label_name, label_idx in [('benign', 0), ('malignant', 1)]:
            label_dir = os.path.join(data_dir, label_name)
            if os.path.exists(label_dir):
                for img_name in os.listdir(label_dir):
                    if img_name.endswith('.png'):
                        self.samples.append((os.path.join(label_dir, img_name), label_idx))
        
        print(f"  Loaded {len(self.samples)} images ({channels}ch)")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert('L')
        
        if self.channels == 3:
            img = Image.merge('RGB', (img, img, img))
        
        if self.transform:
            img = self.transform(img)
        return img, label


class INbreastMultiViewDataset(Dataset):
    """Multi-view dataset (CC+MLO pairs) with channel support."""
    def __init__(self, view_pairs_csv: str, transform=None, channels: int = 3):
        self.transform = transform
        self.channels = channels
        self.samples = []
        
        if os.path.exists(view_pairs_csv):
            df = pd.read_csv(view_pairs_csv)
            for _, row in df.iterrows():
                cc_path = row['cc_path']
                mlo_path = row['mlo_path']
                label = int(row['label_binary'])
                if os.path.exists(cc_path) and os.path.exists(mlo_path):
                    self.samples.append((cc_path, mlo_path, label))
        
        print(f"  Loaded {len(self.samples)} view pairs ({channels}ch)")
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        cc_path, mlo_path, label = self.samples[idx]
        cc_img = Image.open(cc_path).convert('L')
        mlo_img = Image.open(mlo_path).convert('L')
        
        if self.channels == 3:
            cc_img = Image.merge('RGB', (cc_img, cc_img, cc_img))
            mlo_img = Image.merge('RGB', (mlo_img, mlo_img, mlo_img))
        
        if self.transform:
            cc_img = self.transform(cc_img)
            mlo_img = self.transform(mlo_img)
        
        return cc_img, mlo_img, label

print("Dataset classes defined!")

Dataset classes defined!


## 6. Model Loading

In [8]:
def load_single_view_model(model_name: str, checkpoint_path: str, stage: str, config) -> Optional[nn.Module]:
    """Load single-view model (Stage 1-3)."""
    try:
        model = get_model(model_name, num_classes=2)
        checkpoint = torch.load(checkpoint_path, map_location=config.device, weights_only=False)
        
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            elif 'state_dict' in checkpoint:
                model.load_state_dict(checkpoint['state_dict'])
            else:
                model.load_state_dict(checkpoint)
        else:
            model.load_state_dict(checkpoint)
        
        model = model.to(config.device)
        model.eval()
        return model
    except Exception as e:
        print(f"    Error loading {model_name}: {e}")
        return None


def load_stage4_multiview_model(model_name: str, checkpoint_path: str, config) -> Optional[nn.Module]:
    """Load Stage 4 multi-view model (3-channel RGB)."""
    try:
        model = MultiViewWrapper_3ch(
            base_model_name=model_name,
            num_classes=2,
            dropout=0.0,
            img_size=config.img_size_stage4,
            fusion='concat'
        )
        
        checkpoint = torch.load(checkpoint_path, map_location=config.device, weights_only=False)
        
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint:
                model.load_state_dict(checkpoint['model_state_dict'])
            elif 'state_dict' in checkpoint:
                model.load_state_dict(checkpoint['state_dict'])
            else:
                model.load_state_dict(checkpoint)
        else:
            model.load_state_dict(checkpoint)
        
        model = model.to(config.device)
        model.eval()
        return model
    except Exception as e:
        print(f"    Error loading Stage 4 {model_name}: {e}")
        return None

print("Model loading functions defined!")

Model loading functions defined!


## 7. Evaluation Functions

In [9]:
def calculate_metrics(y_true, y_pred, y_prob):
    """Calculate comprehensive metrics."""
    metrics = {'accuracy': accuracy_score(y_true, y_pred) * 100}
    
    try:
        metrics['auc'] = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.5
    except:
        metrics['auc'] = 0.5
    
    try:
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            metrics['sensitivity'] = tp / (tp + fn) * 100 if (tp + fn) > 0 else 0.0
            metrics['specificity'] = tn / (tn + fp) * 100 if (tn + fp) > 0 else 0.0
        else:
            metrics['sensitivity'] = metrics['specificity'] = 0.0
    except:
        metrics['sensitivity'] = metrics['specificity'] = 0.0
    
    try:
        metrics['f1'] = f1_score(y_true, y_pred) * 100
    except:
        metrics['f1'] = 0.0
    
    return metrics


def find_optimal_threshold(y_true, y_prob):
    """Find optimal threshold using Youden's J."""
    try:
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        j_scores = tpr - fpr
        best_idx = np.argmax(j_scores)
        optimal_threshold = thresholds[best_idx]
        y_pred_opt = (y_prob >= optimal_threshold).astype(int)
        return optimal_threshold, calculate_metrics(y_true, y_pred_opt, y_prob)
    except:
        return 0.5, {}


def evaluate_single_view(model, dataloader, config):
    """Evaluate single-view model."""
    model.eval()
    all_probs, all_labels = [], []
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="    Evaluating", ncols=100, leave=False):
            images = images.to(config.device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
    
    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = (y_prob >= 0.5).astype(int)
    
    metrics_default = calculate_metrics(y_true, y_pred, y_prob)
    opt_threshold, metrics_optimal = find_optimal_threshold(y_true, y_prob)
    
    return {'y_true': y_true, 'y_prob': y_prob, 'metrics_default': metrics_default,
            'metrics_optimal': metrics_optimal, 'optimal_threshold': opt_threshold}


def evaluate_multiview(model, dataloader, config):
    """Evaluate multi-view model."""
    model.eval()
    all_probs, all_labels = [], []
    
    with torch.no_grad():
        for cc_imgs, mlo_imgs, labels in tqdm(dataloader, desc="    Evaluating (Multi-View)", ncols=100, leave=False):
            cc_imgs = cc_imgs.to(config.device)
            mlo_imgs = mlo_imgs.to(config.device)
            outputs = model(cc_imgs, mlo_imgs)
            probs = F.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(labels.numpy())
    
    y_true = np.array(all_labels)
    y_prob = np.array(all_probs)
    y_pred = (y_prob >= 0.5).astype(int)
    
    metrics_default = calculate_metrics(y_true, y_pred, y_prob)
    opt_threshold, metrics_optimal = find_optimal_threshold(y_true, y_prob)
    
    return {'y_true': y_true, 'y_prob': y_prob, 'metrics_default': metrics_default,
            'metrics_optimal': metrics_optimal, 'optimal_threshold': opt_threshold}

print("Evaluation functions defined!")

Evaluation functions defined!


## 8. Create Data Loaders

In [10]:
# Transforms for 1-channel models (Stage 2-3)
transform_224_1ch = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Transforms for 3-channel models (Stage 1 - ImageNet)
transform_224_3ch = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Transforms for Stage 4 (768x768, 3-channel, ImageNet normalization)
transform_768_3ch = transforms.Compose([
    transforms.Resize((768, 768)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("\nCreating datasets...")

# Stage 1 dataset (3-channel, ImageNet norm)
print("\n  Stage 1 (224x224, 3ch RGB, ImageNet norm):")
dataset_224_3ch = INbreastDataset(config.inbreast_dir, transform=transform_224_3ch, channels=3)

# Stage 2-3 dataset (1-channel)
print("\n  Stage 2-3 (224x224, 1ch Gray):")
dataset_224_1ch = INbreastDataset(config.inbreast_dir, transform=transform_224_1ch, channels=1)

# Stage 4 multi-view dataset (768x768, 3-channel, ImageNet norm)
print("\n  Stage 4 Multi-View (768x768, 3ch RGB, ImageNet norm):")
dataset_768_3ch_mv = INbreastMultiViewDataset(config.view_pairs_csv, transform=transform_768_3ch, channels=3)

# Create dataloaders
dl_224_3ch = DataLoader(dataset_224_3ch, batch_size=config.batch_size, shuffle=False,
                        num_workers=config.num_workers, pin_memory=True)
dl_224_1ch = DataLoader(dataset_224_1ch, batch_size=config.batch_size, shuffle=False,
                        num_workers=config.num_workers, pin_memory=True)
dl_768_3ch_mv = DataLoader(dataset_768_3ch_mv, batch_size=max(1, config.batch_size // 8),
                           shuffle=False, num_workers=config.num_workers,
                           pin_memory=True) if len(dataset_768_3ch_mv) > 0 else None

print(f"\nDataLoaders created:")
print(f"  Stage 1 (3ch): {len(dl_224_3ch)} batches")
print(f"  Stage 2-3 (1ch): {len(dl_224_1ch)} batches")
print(f"  Stage 4 MV (3ch): {len(dl_768_3ch_mv) if dl_768_3ch_mv else 'NO PAIRS'}")


Creating datasets...

  Stage 1 (224x224, 3ch RGB, ImageNet norm):
  Loaded 410 images (3ch)

  Stage 2-3 (224x224, 1ch Gray):
  Loaded 410 images (1ch)

  Stage 4 Multi-View (768x768, 3ch RGB, ImageNet norm):
  Loaded 2 view pairs (3ch)

DataLoaders created:
  Stage 1 (3ch): 26 batches
  Stage 2-3 (1ch): 26 batches
  Stage 4 MV (3ch): 1


## 9. Run Evaluation

In [11]:
all_results = {}

print("\n" + "=" * 70)
print("RUNNING EVALUATION ON INBREAST DATASET")
print("=" * 70)

for stage, models_dict in MODEL_REGISTRY.items():
    print(f"\n{'='*70}")
    print(f"STAGE: {stage}")
    print(f"{'='*70}")
    
    all_results[stage] = {}
    
    for model_name, model_info in tqdm(models_dict.items(), desc=f"{stage}", ncols=100):
        checkpoint_path = model_info['path']
        is_multiview = model_info['multiview']
        channels = model_info['channels']
        img_size = model_info['img_size']
        
        if not os.path.exists(checkpoint_path):
            print(f"\n  [SKIP] {model_name} - not found")
            continue
        
        print(f"\n  Loading {model_name} ({img_size}x{img_size}, {channels}ch, {'MV' if is_multiview else 'SV'})...")
        
        if stage == 'Stage4' and is_multiview:
            # Stage 4: Multi-view with 3-channel RGB
            if dl_768_3ch_mv is None:
                print(f"    [SKIP] No multi-view pairs available")
                continue
            
            model = load_stage4_multiview_model(model_name, checkpoint_path, config)
            if model is None:
                continue
            
            result = evaluate_multiview(model, dl_768_3ch_mv, config)
        else:
            # Stage 1-3: Single-view
            model = load_single_view_model(model_name, checkpoint_path, stage, config)
            if model is None:
                continue
            
            dataloader = dl_224_3ch if channels == 3 else dl_224_1ch
            result = evaluate_single_view(model, dataloader, config)
        
        all_results[stage][model_name] = result
        
        m = result['metrics_default']
        m_opt = result['metrics_optimal']
        print(f"    Default (t=0.5):  AUC={m['auc']:.4f} | Sens={m['sensitivity']:.1f}% | Spec={m['specificity']:.1f}%")
        if m_opt:
            print(f"    Optimal (t={result['optimal_threshold']:.3f}): Sens={m_opt.get('sensitivity',0):.1f}% | Spec={m_opt.get('specificity',0):.1f}%")
        
        del model
        torch.cuda.empty_cache()

print("\n" + "=" * 70)
print("EVALUATION COMPLETE")
print("=" * 70)


RUNNING EVALUATION ON INBREAST DATASET

STAGE: Stage1


Stage1:   0%|                                                                 | 0/6 [00:00<?, ?it/s]


  Loading ResNet34 (224x224, 3ch, SV)...
Loading pretrained weights for ResNet34...



Stage1:  17%|█████████▌                                               | 1/6 [00:02<00:12,  2.58s/it]

    Default (t=0.5):  AUC=0.4461 | Sens=23.0% | Spec=65.8%
    Optimal (t=0.834): Sens=11.0% | Spec=90.3%

  Loading ResNet50 (224x224, 3ch, SV)...
Loading pretrained weights for ResNet50...


...

    Default (t=0.5):  AUC=0.4272 | Sens=0.0% | Spec=100.0%
    Optimal (t=0.165): Sens=20.0% | Spec=83.2%

STAGE: Stage2


Stage2:   0%|                                                                 | 0/3 [00:00<?, ?it/s]


  Loading ResNet50Stage2 (224x224, 1ch, SV)...
Loading pretrained weights for ResNet50Stage2...



Stage2:   0%|                                                                 | 0/3 [00:00<?, ?it/s]


RuntimeError: Given groups=1, weight of size [64, 3, 7, 7], expected input[16, 1, 224, 224] to have 3 channels, but got 1 channels instead

## 10. Results Table

In [ ]:
results_data = []
for stage, stage_results in all_results.items():
    for model_name, result in stage_results.items():
        m = result['metrics_default']
        m_opt = result.get('metrics_optimal', {})
        results_data.append({
            'Stage': stage, 'Model': model_name,
            'AUC': m.get('auc', 0),
            'Sens (%)': m.get('sensitivity', 0),
            'Spec (%)': m.get('specificity', 0),
            'Acc (%)': m.get('accuracy', 0),
            'Opt Thresh': result.get('optimal_threshold', 0.5),
            'Opt Sens': m_opt.get('sensitivity', 0) if m_opt else 0,
            'Opt Spec': m_opt.get('specificity', 0) if m_opt else 0,
        })

results_df = pd.DataFrame(results_data)
print("\nRESULTS SUMMARY")
print("=" * 100)
print(results_df.to_string(index=False))

results_df.to_csv(os.path.join(config.output_dir, 'inbreast_results.csv'), index=False)
print(f"\nSaved to: {config.output_dir}")

## 11. Visualizations

In [ ]:
# ROC Curves
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()
colors = plt.cm.tab10.colors

for idx, (stage, stage_results) in enumerate(all_results.items()):
    ax = axes[idx]
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    
    for i, (model_name, result) in enumerate(stage_results.items()):
        try:
            fpr, tpr, _ = roc_curve(result['y_true'], result['y_prob'])
            auc = result['metrics_default']['auc']
            ax.plot(fpr, tpr, color=colors[i % 10], lw=2, label=f"{model_name} ({auc:.3f})")
        except:
            pass
    
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.set_title(f'{stage} ROC Curves')
    ax.legend(loc='lower right', fontsize=8)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(os.path.join(config.output_dir, 'roc_curves.png'), dpi=150)
plt.show()

In [ ]:
# AUC Comparison
if len(results_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    stages = list(all_results.keys())
    all_models = sorted(set(results_df['Model']))
    x = np.arange(len(all_models))
    width = 0.2
    stage_colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']
    
    for i, stage in enumerate(stages):
        aucs = []
        for model in all_models:
            match = results_df[(results_df['Stage'] == stage) & (results_df['Model'] == model)]
            aucs.append(match['AUC'].values[0] if len(match) > 0 else 0)
        offset = (i - len(stages)/2 + 0.5) * width
        ax.bar(x + offset, aucs, width, label=stage, color=stage_colors[i], alpha=0.8)
    
    ax.set_xlabel('Model')
    ax.set_ylabel('AUC')
    ax.set_title('AUC Comparison Across Stages (INbreast)')
    ax.set_xticks(x)
    ax.set_xticklabels(all_models, rotation=45, ha='right')
    ax.legend()
    ax.set_ylim([0, 1])
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(os.path.join(config.output_dir, 'auc_comparison.png'), dpi=150)
    plt.show()

## 12. Analysis

In [ ]:
print("\n" + "=" * 70)
print("ANALYSIS")
print("=" * 70)

# Best per stage
print("\nBest Model per Stage:")
for stage in all_results:
    if all_results[stage]:
        best = max(all_results[stage].items(), key=lambda x: x[1]['metrics_default']['auc'])
        print(f"  {stage}: {best[0]} (AUC={best[1]['metrics_default']['auc']:.4f})")

# Overall best
all_flat = [(s, n, r['metrics_default']['auc']) for s, sr in all_results.items() for n, r in sr.items()]
if all_flat:
    best = max(all_flat, key=lambda x: x[2])
    print(f"\nOverall Best: {best[0]} - {best[1]} (AUC={best[2]:.4f})")

# Save report
report = {
    'timestamp': datetime.now().isoformat(),
    'results': {s: {n: {'auc': r['metrics_default']['auc'], 'sens': r['metrics_default']['sensitivity'],
                       'spec': r['metrics_default']['specificity']} for n, r in sr.items()}
               for s, sr in all_results.items()}
}
with open(os.path.join(config.output_dir, 'report.json'), 'w') as f:
    json.dump(report, f, indent=2)

print(f"\nReport saved to: {config.output_dir}")